In [ ]:
import functools
import json
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, RegExp as TypeScriptRegExp, TypeScript5, Types as TypeScriptTypes
from mbse.Expressions.TypeScript.Patterns import Expressions as TSP
from mbse.Expressions.TypeScript.RegExp import Expressions as TSR, Text as TSRText
from mbse.Expressions.TypeScript.TypeScript5 import Evaluators as TV, Expressions as TS, Text as TSText
from mbse.Expressions.TypeScript.Types import Expressions as TST
from mbse.Expressions.Framework import Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()  # the framework, the standard dialects TypeScript5's domains use, and TypeScript's family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Unicode.install, TypeScript5.install,
                TypeScriptTypes.install, TypeScriptPatterns.install, TypeScriptRegExp.install]:
    store.transaction(install)
store.transaction(TypeScriptRegExp.define)  # the editions of ECMA-262 the RegExp dialect covers


def held(spec, dialect=TSR.DIALECT):
    """The holder of the term `spec` writes, `dialect`'s own (the RegExp dialect's unless given), written in a
    transaction."""
    with store.transaction():
        return functools.reduce(getattr, dialect.path.split("."), store.Expressions).Expression(
            lambda e: e.root(dialect.spec(spec)))


def regexp(spec):
    """A term of the RegExp dialect, held by its holder."""
    return F.term(held(spec))


def ts(spec):
    """A term of TypeScript5, held by its holder."""
    return F.term(held(spec, TS.DIALECT))


def char(text, raw=None):
    """A spec of the character `text` (or a code point), with its spelling where given."""
    value = text if type(text) is int else ord(text)
    return lambda e: e.character(lambda c: (c if raw is None else c.raw(raw)).value(value))


def seq(*elements):
    """A spec of an alternative of `elements`."""
    return lambda e: e.alternative(lambda a: a.elements(list(elements)))


def pattern(*alternatives):
    """A spec of a pattern of `alternatives`."""
    return lambda e: e.pattern(lambda p: p.alternatives(list(alternatives)))


# /(?i:a)(?<n>b+?)[a-c\d][\q{xy|z}&&[xy]][\w--a]\p{L}.\b(?=c)\k<n>|$/v: a literal of every kind
every = lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq(  # noqa: E731
    lambda e: e.group(lambda g: g.modifiers(lambda m: m.modifiers(lambda s: s.add(lambda a: a.ignoreCase(True)))).alternatives([
        seq(char("a"))])),
    lambda e: e.capturinggroup(lambda g: g.name("n").alternatives([seq(lambda e: e.quantifier(lambda q: q.min(1).greedy(False).element(
        char("b"))))])),
    lambda e: e.characterclass(lambda c: c.unicodeSets(True).elements([
        lambda e: e.characterclassrange(lambda r: r.min(char("a")).max(char("c"))), lambda e: e.characterset(lambda s: s.setKind("digit"))])),
    lambda e: e.expressioncharacterclass(lambda x: x.expression(lambda e: e.classintersection(lambda i: i.left(
        lambda e: e.classstringdisjunction(lambda d: d.alternatives([lambda e: e.stringalternative(lambda s: s.elements([char("x"), char("y")])),
                                                                     lambda e: e.stringalternative(lambda s: s.elements([char("z")]))]))).right(
        lambda e: e.characterclass(lambda c: c.unicodeSets(True).elements([char("x"), char("y")])))))),
    lambda e: e.expressioncharacterclass(lambda x: x.expression(lambda e: e.classsubtraction(lambda s: s.left(
        lambda e: e.characterset(lambda t: t.setKind("word"))).right(char("a"))))),
    lambda e: e.characterset(lambda s: s.raw("\\p{L}").setKind("property").strings(False).key("General_Category").value("L")),
    lambda e: e.characterset(lambda s: s.setKind("any")),
    lambda e: e.assertion(lambda a: a.assertionKind("word")),
    lambda e: e.assertion(lambda a: a.assertionKind("lookahead").alternatives([seq(char("c"))])),
    lambda e: e.backreference(lambda b: b.ref("n"))), seq(lambda e: e.assertion(lambda a: a.assertionKind("end"))))).flags(
    lambda f: f.unicodeSets(True)))
EVERY = "/(?i:a)(?<n>b+?)[a-c\\d][\\q{xy|z}&&[xy]][\\w--a]\\p{L}.\\b(?=c)\\k<n>|$/v"

# Suite TSR — The TypeScript RegExp dialect

ECMA-262's regular expressions (22.2.1, "Patterns"), the sublanguage a regular expression literal holds (12.9.5): the
dialect `Expressions.TypeScript.RegExp` of the TypeScript family, its kinds regexpp's node types, its union, its holder,
its completeness against regexpp's node types kept as data (`grammars/ecmascript-regexpp.json`), its checks, its
editions of ECMA-262, its printing, its evaluation, which refuses each kind by name, and TypeScript5's `constant`, which
holds a literal's parsed regular expression beside its spelling (the dialects proposal, Retrofit's design; mbse-basis
[CAP-226: complete expression dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-226.md)).

In [ ]:
# Its Term: regexpp's node types but its two helpers, and invalid syntax; each kind's and helper's schema is named by
# regexpp's name, under its path
union = [b.name for b in store.registered(TSR.DIALECT.union).branches]
assert union == [*TSR.TAGS, "error", "missing"] and TSR.HELPERS == ("Flags", "ModifierFlags")
assert TSR.DIALECT.name() == "TypeScript RegExp" and TSR.DIALECT.tag == "typescript_regexp"
assert TSR.DIALECT.path == "TypeScript.RegExp" and TSR.DIALECT.union == "Expressions.TypeScript.RegExp.Term"
RegExp = store.registered(TSR.DIALECT.union)
kinds = {b.name: b.schema for b in RegExp.branches}
NAMES = ("RegExpLiteral", "Pattern", "Alternative", "Group", "CapturingGroup", "Quantifier", "CharacterClass",
         "CharacterClassRange", "ClassIntersection", "ClassSubtraction", "ClassStringDisjunction", "StringAlternative",
         "ExpressionCharacterClass", "CharacterSet", "Character", "Assertion", "Backreference", "Modifiers")
assert [kinds[tag].name for tag in TSR.TAGS] == [f"Expressions.TypeScript.RegExp.{name}" for name in NAMES]
assert [name.lower() for name in NAMES] == list(TSR.TAGS)  # each regexpp's node type, lowercased
assert {tag: list(kinds[tag].properties) for tag in TSR.TAGS} == {
    "regexpliteral": ["pattern", "flags"], "pattern": ["alternatives"], "alternative": ["elements"],
    "group": ["modifiers", "alternatives"], "capturinggroup": ["name", "alternatives"],
    "quantifier": ["raw", "min", "max", "greedy", "element"], "characterclass": ["unicodeSets", "negate", "elements"],
    "characterclassrange": ["min", "max"], "classintersection": ["left", "right"], "classsubtraction": ["left", "right"],
    "classstringdisjunction": ["alternatives"], "stringalternative": ["elements"],
    "expressioncharacterclass": ["negate", "expression"],
    "characterset": ["raw", "setKind", "negate", "strings", "key", "value"], "character": ["raw", "value"],
    "assertion": ["assertionKind", "negate", "alternatives"], "backreference": ["raw", "ref"], "modifiers": ["add", "remove"]}
assert F._parts(kinds["regexpliteral"]) == (("pattern",), (), ("flags",)) and F._helpers(kinds["regexpliteral"]) == ("flags",)
assert F._parts(kinds["group"]) == (("modifiers",), ("alternatives",), ())  # its modifiers a term, `modifiers`
assert F._parts(kinds["quantifier"]) == (("element",), (), ("raw", "min", "max", "greedy"))
assert F._parts(kinds["modifiers"]) == ((), (), ("add", "remove")) and F._helpers(kinds["modifiers"]) == ("add", "remove")
assert list(kinds["backreference"].properties["ref"].schema.schemas) == [store.registered("Schemas.OfInt"),
                                                                          store.registered("Schemas.OfStr")]  # a number or a name
# Its helpers, value objects its kinds hold: a literal's flags, a modifiers' flags, each its spelling and its booleans
helpers = {name: store.registered(f"Expressions.TypeScript.RegExp.{name}") for name in TSR.HELPERS}
assert {name: (h.ref, list(h.properties)) for name, h in helpers.items()} == {
    "Flags": (False, ["raw", "dotAll", "isGlobal", "hasIndices", "ignoreCase", "multiline", "sticky", "unicode", "unicodeSets"]),
    "ModifierFlags": (False, ["raw", "dotAll", "ignoreCase", "multiline"])}
assert kinds["regexpliteral"].properties["flags"].schema is helpers["Flags"]
assert kinds["modifiers"].properties["remove"].schema is helpers["ModifierFlags"]
assert [letter for _, letter in TSR.FLAGS] == list("dgimsuvy") and [letter for _, letter in TSR.MODIFIERS] == list("ims")
# None refines Let: a named group's name is a label its backreferences resolve, not a name a term binds
assert not any(F.is_binder(kinds[tag]) for tag in TSR.TAGS)
# Where it is held: TypeScript5's constant, its `regexp`, and nowhere else in the family
held_at = [(dialect.name(), tag, name) for dialect in (TS.DIALECT, TST.DIALECT, TSP.DIALECT)
           for tag, kind in ((b.name, b.schema) for b in store.registered(dialect.union).branches)
           for name, prop in kind.properties.items() if prop.schema is RegExp]
assert held_at == [("TypeScript5", "constant", "regexp")]
# Its holder, refining the framework's Expression, its root narrowed to its Term: validated, linked by HasRoot,
# round-tripped
Holder = store.registered("Expressions.TypeScript.RegExp.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is RegExp and list(Holder.adjacencies) == ["abi"] and Holder.validate() == []
literal = held(every)
assert Validators.Validate(store)(Holder, literal) == [] and TSR.DIALECT.validate(literal) == []
with store.transaction():
    linked = store.Schemas.OfExpression(lambda s: s.name("tsr.every").expression(lambda a: a.insert(lambda e: e.expression(literal))))
assert [F.same(entry.expression, literal) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(decode(Holder, encode(Holder, literal)).root, literal.root)
    assert F.same(decode(RegExp, encode(RegExp, F.term(literal))), F.term(literal))
# Its install runs before or after its family's: TypeScript5's names its Term, for a constant's regexp, either creating it
for order in ([TypeScriptRegExp.install, TypeScript5.install], [TypeScript5.install, TypeScriptRegExp.install]):
    alone = Builders.Store()
    for install in [Framework.install, *order, TypeScriptTypes.install, TypeScriptPatterns.install]:
        alone.transaction(install)
    assert [b.name for b in alone.registered(TSR.DIALECT.union).branches] == union
    constant = {b.name: b.schema for b in alone.registered(TS.DIALECT.union).branches}["constant"]
    assert constant.properties["regexp"].schema is alone.registered(TSR.DIALECT.union)

<a id="TSR-01"></a>
## TSR-01 · The RegExp dialect's union, its kinds' and helpers' schemas in regexpp's shapes, the one position that holds it, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
grammar = json.loads(Path("../../grammars/ecmascript-regexpp.json").read_text(encoding="utf-8"))  # regexpp's node types, as data
assert grammar["checked"].startswith("@eslint-community/regexpp 4.12.3 (ECMAScript 2025, its latest): its 20 node types")
nodes = grammar["nodes"]
assert len(nodes) == len({node["node"] for node in nodes}) == 20  # every node type, once
mapped, variants = {"kinds": [], "helpers": []}, {}
for node in nodes:  # each a kind or a helper, its properties the schema's, in regexpp's order, renamed where the rule says
    ways = [way for way in ("kinds", "helpers") if way in node]
    assert len(ways) == 1 and len(node[ways[0]]) == 1 and set(node) <= {"node", *ways, "interfaces", "properties", "dropped"}, node
    name = node[ways[0]][0]
    mapped[ways[0]].append(name)
    if ways == ["kinds"]:  # a branch of its Term, its type lowercased
        assert name == node["node"].lower() and name in kinds, node
        schema = kinds[name]
    else:  # a value object schema it registers, not a branch
        assert name == node["node"] and name.lower() not in kinds, node
        schema = helpers[name]
    assert list(schema.properties) == list(node["properties"].values()), node
    assert {k: v for k, v in node["properties"].items() if k != v} == (
        {"kind": "assertionKind"} if name == "assertion" else {"kind": "setKind"} if name == "characterset"
        else {"global": "isGlobal"} if name == "Flags" else {}), node  # a keyword of Python or the framework's `kind`
    assert set(node.get("dropped", ())) <= {"raw", "references", "resolved", "ambiguous"}, node  # what regexpp derives
    assert ("raw" in node["properties"]) != ("raw" in node.get("dropped", ())), node  # every node has its spelling, kept or not
    variants[name] = {kind for found in node.get("interfaces", {}).values() for kind in found}
assert sorted(mapped["kinds"]) == sorted(TSR.TAGS) and mapped["helpers"] == list(TSR.HELPERS)
# A leaf keeps its spelling, and a quantifier its own; every other node is spelled by its parts
assert [node["node"] for node in nodes if "raw" in node["properties"]] == [
    "Quantifier", "Backreference", "Character", "CharacterSet", "Flags", "ModifierFlags"]
# A type several interfaces share: the kinds of each are the ones its checks accept
assert variants["assertion"] == set(TSR.ASSERTIONS) and variants["characterset"] == set(TSR.CHARACTERSETS)
assert [n["node"] for n in nodes if "interfaces" in n] == ["CharacterClass", "Assertion", "Backreference", "CharacterSet"]

<a id="TSR-02"></a>
## TSR-02 · It is complete: each of regexpp's node types, kept as data, is a kind or a helper, its properties the schema's, its interfaces' kinds those its checks accept

In [ ]:
a, z = char("a"), char("z")
word = lambda e: e.characterset(lambda s: s.setKind("word"))  # noqa: E731
for spec, problems in [
    # a literal: its pattern, and its flags: each once, ECMA-262's, as its spelling spells them, not both u and v
    (lambda r: r.regexpliteral(lambda l: l), ["a regexpliteral needs a pattern"]),
    (lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq())).flags(lambda f: f.raw("gg").isGlobal(True))),
     ["a regular expression's flags cannot be 'gg'"]),
    (lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq())).flags(lambda f: f.raw("g").ignoreCase(True))),
     ["a regular expression's flags 'g' do not spell the flags it holds"]),
    (lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq())).flags(lambda f: f.unicode(True).unicodeSets(True))),
     ["a regular expression's flags cannot have both u and v"]),
    # a disjunction needs an alternative (an empty one, where it is empty); a group's name, where given, is not empty
    (lambda r: r.pattern(lambda p: p), ["a pattern needs an alternative"]),
    (lambda r: r.group(lambda g: g), ["a group needs an alternative"]),
    (lambda r: r.capturinggroup(lambda g: g.name("")), ["a capturinggroup's name cannot be empty", "a capturinggroup needs an alternative"]),
    (lambda r: r.classstringdisjunction(lambda d: d), ["a classstringdisjunction needs an alternative"]),
    (lambda r: r.assertion(lambda s: s.assertionKind("lookbehind")), ["an assertion needs an alternative"]),
    # a quantifier: its element, its bounds in order
    (lambda r: r.quantifier(lambda q: q.min(0)), ["a quantifier needs an element"]),
    (lambda r: r.quantifier(lambda q: q.element(a)), ["a quantifier needs a min"]),
    (lambda r: r.quantifier(lambda q: q.min(-1).element(a)), ["a quantifier's min must not be negative, got -1"]),
    (lambda r: r.quantifier(lambda q: q.min(2).max(1).element(a)), ["a quantifier's max 1 is less than its min 2"]),
    (lambda r: r.quantifier(lambda q: q.min("2").element(a)), ["a quantifier's min must be a int, got str"]),  # its type, by validation
    # a range: of characters, in order
    (lambda r: r.characterclassrange(lambda c: c.min(z).max(a)), ["a characterclassrange's min 122 is greater than its max 97"]),
    (lambda r: r.characterclassrange(lambda c: c.min(word).max(lambda e: e.error(lambda x: x.text("\\")))),
     ["a characterclassrange's min must be a character, got characterset", "max: invalid syntax '\\\\'"]),
    (lambda r: r.characterclassrange(lambda c: c.max(a)), ["a characterclassrange needs a min"]),
    # a class's expression: an intersection or a subtraction
    (lambda r: r.expressioncharacterclass(lambda x: x.expression(a)),
     ["an expressioncharacterclass's expression must be a classintersection or a classsubtraction, got character"]),
    (lambda r: r.expressioncharacterclass(lambda x: x), ["an expressioncharacterclass needs an expression"]),
    # a character set: its kind, a property's key, what each kind has
    (lambda r: r.characterset(lambda s: s), ["a characterset needs a setKind"]),
    (lambda r: r.characterset(lambda s: s.setKind("hex")), ["CharacterSet.setKind cannot be 'hex'"]),
    (lambda r: r.characterset(lambda s: s.setKind("property").value("L")), ["a property characterset needs a key"]),
    (lambda r: r.characterset(lambda s: s.setKind("property").key("RGI_Emoji").strings(True).negate(True)),
     ["a characterset of strings cannot be negated"]),
    (lambda r: r.characterset(lambda s: s.setKind("digit").key("L")), ["a digit characterset has no key, value or strings"]),
    (lambda r: r.characterset(lambda s: s.setKind("any").negate(True)), ["an any characterset cannot be negated"]),
    # a character: a code point
    (lambda r: r.character(lambda c: c.raw("a")), ["a character needs a value"]),
    (lambda r: r.character(lambda c: c.value(0x110000)), ["a character's value must be a code point, got 1114112"]),
    (lambda r: r.character(lambda c: c.value("a")), ["a character's value must be a int, got str"]),  # its type, by validation
    # an assertion: its kind; a lookaround's alternatives, and only a lookaround's or a word boundary's negation
    (lambda r: r.assertion(lambda s: s), ["an assertion needs an assertionKind"]),
    (lambda r: r.assertion(lambda s: s.assertionKind("ahead")), ["Assertion.assertionKind cannot be 'ahead'"]),
    (lambda r: r.assertion(lambda s: s.assertionKind("start").alternatives([seq(a)])), ["a start assertion holds no alternatives"]),
    (lambda r: r.assertion(lambda s: s.assertionKind("end").negate(True)), ["an end assertion cannot be negated"]),
    # a backreference: a group's number or its name
    (lambda r: r.backreference(lambda b: b), ["a backreference needs a ref"]),
    (lambda r: r.backreference(lambda b: b.ref(0)), ["a backreference's ref must be a group's number or its name, got 0"]),
    (lambda r: r.backreference(lambda b: b.ref("")), ["a backreference's ref must be a group's number or its name, got ''"]),
    # modifiers: each of `ims` once, a flag added or removed, none both
    (lambda r: r.modifiers(lambda m: m.add(lambda f: f.raw("x"))), ["a modifiers' flags cannot be 'x'"]),
    (lambda r: r.modifiers(lambda m: m.add(lambda f: f.raw(""))), ["a modifiers adds or removes a flag"]),
    (lambda r: r.modifiers(lambda m: m.add(lambda f: f.ignoreCase(True)).remove(lambda f: f.raw("im").ignoreCase(True).multiline(True))),
     ["a modifiers cannot both add and remove 'i'"]),
    # invalid syntax within a kind, by its container
    (lambda r: r.alternative(lambda s: s.elements([a, lambda e: e.missing(lambda m: m.kind("Atom"))])), ["argument 1: missing Atom"]),
]:
    assert TSR.DIALECT.validate(regexp(spec)) == problems, (TSR.DIALECT.validate(regexp(spec)), problems)
# What its checks accept: a group's modifiers that remove alone, a lazy quantifier, a range in order, a property of strings
for spec in [lambda r: r.group(lambda g: g.modifiers(lambda m: m.modifiers(lambda s: s.remove(lambda f: f.raw("s").dotAll(True)))).alternatives([seq()])),
             lambda r: r.quantifier(lambda q: q.min(2).max(2).greedy(False).element(a)),
             lambda r: r.characterclassrange(lambda c: c.min(a).max(z)), lambda r: r.backreference(lambda b: b.ref(1)),
             lambda r: r.characterset(lambda s: s.setKind("property").key("RGI_Emoji").strings(True)),
             lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq())).flags(lambda f: f.raw("dgimsuy").hasIndices(True).isGlobal(
                 True).ignoreCase(True).multiline(True).dotAll(True).unicode(True).sticky(True))), every]:
    assert TSR.DIALECT.validate(regexp(spec)) == []

<a id="TSR-03"></a>
## TSR-03 · Each kind's checks: ECMA-262's early errors its data can show, and what a kind needs

In [ ]:
# Its kinds read and written by their forms, each by the dialect whose union holds it, in its holder: its helpers'
# parts among its attributes, its terms its arguments
walked = list(F.walk(regexp(every)))
assert {F.kind_of(node)[0] for node in walked} == set(TSR.TAGS)  # every kind
for node in walked:
    assert F.same(F.term(held(TSR.DIALECT.make(F.form(node)))), node)
form = F.form(regexp(every))
assert form.kind == "regexpliteral" and form.attributes == {"flags": F.Part("flags", {"unicodeSets": True}, 0)}
assert [F.kind_of(argument)[0] for argument in form.arguments] == ["pattern"]
quantifier = next(node for node in walked if F.kind_of(node)[0] == "quantifier")
assert F.form(quantifier).attributes == {"min": 1, "greedy": False}
modifiers = next(node for node in walked if F.kind_of(node)[0] == "modifiers")
assert F.form(modifiers) == F.Form("modifiers", {"add": F.Part("modifierflags", {"ignoreCase": True}, 0)})
with raises(ValueError, match="TypeScript RegExp has no kind 'call'"):  # neither it nor a dialect it reads has one
    TSR.DIALECT.make(F.Form("call"))
# It binds nothing: a named group's name is a label, which a backreference resolves; no name is free within it
assert SY.free(regexp(every), TSR.DIALECT) == set() and TSR.DIALECT.validate(regexp(lambda r: r.backreference(lambda b: b.ref("x")))) == []
# It reads its family's other dialects, and each of them reads it; a regular expression's kinds are its own
assert TSR.DIALECT.reads() == (TS.DIALECT, TST.DIALECT, TSP.DIALECT) and TS.DIALECT.reads()[-1] is TSR.DIALECT
assert TS.DIALECT.kind_named("character") is TSR.DIALECT.kinds()["character"] and TS.DIALECT.owner(regexp(every)) is TSR.DIALECT
assert list(TSR.DIALECT.kinds()) == [*TSR.TAGS, "error", "missing"]
# Evaluation refuses each kind by its dialect's name, as TypeScript5's evaluator does; inference knows nothing of it
for node in walked:
    with raises(NotImplementedError, match=f"TypeScript RegExp {F.kind_of(node)[0]} is not evaluated"):
        TV.OfAny(node)
with raises(ValueError, match="TypeScript RegExp error is not evaluated"):
    TV.OfAny(regexp(lambda r: r.error(lambda e: e.text("("))))
assert TSR.DIALECT.infer(regexp(every)).name() == "any"

<a id="TSR-04"></a>
## TSR-04 · Forms and `make`; it binds nothing and reads its family's dialects, as they read it; evaluation refuses each kind by name

In [ ]:
tsr = store.Instances.Expressions.TypeScript.RegExp
DESIGNATIONS = ("ECMA-262 5.1 Edition", *(f"ECMA-262 {n}th Edition" for n in range(6, 17)))
assert tuple(FS.standards(tsr)) == DESIGNATIONS == tuple(edition[0] for edition in TSR.EDITIONS)
assert FS.properties(FS.find(tsr, "ES5.1")) == {"bodies": ("org.ecma-international",), "number": "262",
                                                "revision": ("version", "5.1"), "names": ("ECMAScript 5.1", "ES5.1")}
assert FS.properties(FS.find(tsr, "ECMAScript 2015")) == {"bodies": ("org.ecma-international",), "number": "262",
                                                          "revision": ("edition", 6), "names": ("ECMAScript 2015", "ES2015")}
assert FS.tags(FS.find(tsr, "ES2025")) == ("ECMA-262 16th Edition", "ECMAScript 2025", "ES2025")  # found by any of its tags
assert FS.find(tsr, "ES2026") is None and FS.find(tsr, "TypeScript 5.0") is None  # its own editions alone
# Each edition its kinds: every one but the `v` flag's five, from the 15th (ES2024), and modifiers, from the 16th (ES2025)
V = {"classintersection", "classsubtraction", "classstringdisjunction", "stringalternative", "expressioncharacterclass"}
assert TSR.SINCE == {**dict.fromkeys(V, 10), "modifiers": 11}
counts = [len(FS.kinds(standard)) for standard in FS.standards(tsr).values()]
assert counts == [12] * 10 + [17, 18]
assert [s.name for s in FS.kinds(FS.find(tsr, "ES2025"))] == [kinds[tag].name for tag in TSR.TAGS]  # its own kinds alone
assert tuple(FS.tags(s)[-1] for s in FS.of(tsr, kinds["modifiers"]).values()) == ("ES2025",)
assert tuple(FS.tags(s)[-1] for s in FS.of(tsr, kinds["classintersection"]).values()) == ("ES2024", "ES2025")
assert len(FS.of(tsr, kinds["character"])) == 12 and FS.of(tsr, store.registered("Expressions.Framework.Error")) == {}

<a id="TSR-05"></a>
## TSR-05 · Its editions: ECMA-262's, the 5.1 Edition to the 16th, by Ecma International's designations and their common names, each with the kinds it has

In [ ]:
assert TSRText.ToText is TSText.ToText and set(TSRText.TEXTS) == set(TSR.TAGS)
assert TSText.ToText(held(every)) == EVERY  # every kind, from its own holder
group = lambda m: lambda r: r.group(lambda g: g.modifiers(lambda e: e.modifiers(m)).alternatives([seq(char("a"))]))  # noqa: E731
quantified = lambda q: lambda r: r.quantifier(lambda s: q(s).element(char("a")))  # noqa: E731
in_class = lambda *elements: lambda r: r.characterclass(lambda c: c.elements(list(elements)))  # noqa: E731
for spec, text in [
    # a spelling as written: a character's, a character set's, a backreference's, a literal's and a modifiers' flags, a
    # quantifier's where it begins with its element's text
    (char("A", "\\x41"), "\\x41"), (lambda r: r.characterset(lambda s: s.raw("\\p{gc=L}").setKind("property").key("gc").value("L")), "\\p{gc=L}"),
    (lambda r: r.backreference(lambda b: b.raw("\\k<n>").ref("n")), "\\k<n>"),
    (lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq(char("a")))).flags(lambda f: f.raw("ig").isGlobal(True).ignoreCase(True))), "/a/ig"),
    (group(lambda s: s.add(lambda f: f.raw("mi").ignoreCase(True).multiline(True))), "(?mi:a)"),
    (quantified(lambda q: q.raw("a{0,}").min(0)), "a{0,}"), (quantified(lambda q: q.raw("b{0,}").min(0)), "a*"),
    # else its default spelling: a quantifier's shortest, lazy where it is not greedy
    (quantified(lambda q: q.min(1)), "a+"), (quantified(lambda q: q.min(0).max(1)), "a?"),
    (quantified(lambda q: q.min(2)), "a{2,}"), (quantified(lambda q: q.min(3).max(3)), "a{3}"),
    (quantified(lambda q: q.min(2).max(5).greedy(False)), "a{2,5}?"), (quantified(lambda q: q.min(0).greedy(False)), "a*?"),
    # a character: itself, a syntax character and `/` escaped, in a class `-` too, a control or a line terminator escaped
    (lambda r: r.alternative(lambda s: s.elements([char(c) for c in "^$\\.*+?()[]{}|/-é"] + [char(0x1F600)])),
     "\\^\\$\\\\\\.\\*\\+\\?\\(\\)\\[\\]\\{\\}\\|\\/-é😀"),
    (in_class(char("-"), char("]"), char("^"), lambda e: e.characterclassrange(lambda c: c.min(char("-")).max(char("/")))),
     "[\\-\\]\\^\\--\\/]"),
    (lambda r: r.alternative(lambda s: s.elements([char(c) for c in (9, 10, 11, 12, 13, 0, 0x1F, 0x7F, 0x2028, 0x2029, 0xD800)])),
     "\\t\\n\\v\\f\\r\\x00\\x1F\\x7F\\u2028\\u2029\\uD800"),
    # a character set, an assertion, a group, a backreference, a class, a string disjunction, flags
    (lambda r: r.alternative(lambda s: s.elements([lambda e, kind=kind, negate=negate: e.characterset(lambda c: c.setKind(kind).negate(negate))
                                                    for kind in ("digit", "space", "word") for negate in (False, True)])),
     "\\d\\D\\s\\S\\w\\W"),
    (lambda r: r.characterset(lambda s: s.setKind("property").key("Script").value("Greek").negate(True)), "\\P{Script=Greek}"),
    (lambda r: r.characterset(lambda s: s.setKind("property").key("RGI_Emoji").strings(True)), "\\p{RGI_Emoji}"),
    (lambda r: r.alternative(lambda s: s.elements([lambda e, kind=kind, negate=negate: e.assertion(lambda a: a.assertionKind(kind).negate(negate).alternatives(
        [seq(char("a"))] if kind.startswith("look") else [])) for kind, negate in (
        ("start", False), ("end", False), ("word", False), ("word", True), ("lookahead", False), ("lookahead", True),
        ("lookbehind", False), ("lookbehind", True))])), "^$\\b\\B(?=a)(?!a)(?<=a)(?<!a)"),
    (lambda r: r.group(lambda g: g.alternatives([seq(char("a")), seq()])), "(?:a|)"),
    (group(lambda s: s.remove(lambda f: f.multiline(True))), "(?-m:a)"),
    (group(lambda s: s.add(lambda f: f.ignoreCase(True).dotAll(True)).remove(lambda f: f.multiline(True))), "(?is-m:a)"),
    (lambda r: r.capturinggroup(lambda g: g.alternatives([seq(char("a"))])), "(a)"),
    (lambda r: r.backreference(lambda b: b.ref(12)), "\\12"),
    (lambda r: r.characterclass(lambda c: c.negate(True).elements([char("a"), char("A", "\\x41")])), "[^a\\x41]"),
    (lambda r: r.expressioncharacterclass(lambda x: x.negate(True).expression(lambda e: e.classsubtraction(lambda s: s.left(
        char("-")).right(lambda e: e.classstringdisjunction(lambda d: d.alternatives([lambda e: e.stringalternative(
            lambda s: s.elements([char("|"), char("-")]))])))))), "[^\\---\\q{\\|\\-}]"),
    (lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq(char("/")))).flags(lambda f: f.sticky(True).hasIndices(True).isGlobal(
        True).ignoreCase(True).multiline(True).dotAll(True).unicodeSets(True))), "/\\//dgimsvy"),
    (lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq()))), "//"),  # no flags; an empty pattern, as it holds it
    # invalid syntax, as written
    (lambda r: r.alternative(lambda s: s.elements([char("a"), lambda e: e.error(lambda x: x.text("(?")),
                                                   lambda e: e.missing(lambda m: m.kind("Atom"))])), "a(?"),
]:
    assert TSText.ToText(regexp(spec)) == text, (TSText.ToText(regexp(spec)), text)
with raises(ValueError, match="TypeScript RegExp error has no text to print"):  # refused by its dialect's name
    TSText.ToText(regexp(lambda r: r.error(lambda e: e)))

<a id="TSR-06"></a>
## TSR-06 · Printed as ECMAScript writes a regular expression, by the family's printer: every kind, a spelling as written, else its default spelling

In [ ]:
# Its schema: its spelling, `pattern` and `flags`, then `regexp`, the regular expression they encode, a slot holding the
# RegExp dialect's Term
Constant = {b.name: b.schema for b in store.registered(TS.DIALECT.union).branches}["constant"]
assert list(Constant.properties) == ["singleton", "pattern", "flags", "regexp", "text", "problems"]
assert Constant.properties["regexp"].schema is RegExp
assert F._parts(Constant) == (("regexp",), (), ("singleton", "pattern", "flags", "text", "problems"))
slash = lambda r: r.regexpliteral(lambda l: l.pattern(pattern(seq(char("a")))).flags(lambda f: f.raw("g").isGlobal(True)))  # noqa: E731
spelled = ts(lambda e: e.constant(lambda c: c.pattern("a").flags("g").regexp(slash)))  # /a/g, read by a parser
assert TS.DIALECT.validate(spelled) == [] and TSText.ToText(spelled) == "/a/g"
assert F.form(spelled).attributes == {"pattern": "a", "flags": "g"} and [F.kind_of(a)[0] for a in F.form(spelled).arguments] == ["regexpliteral"]
assert [F.kind_of(node)[0] for node in F.walk(spelled)] == ["constant", "regexpliteral", "pattern", "alternative", "character"]
assert F.same(F.term(held(TS.DIALECT.make(F.form(spelled)), TS.DIALECT)), spelled)
assert SY.free(spelled, TS.DIALECT) == set() and TS.DIALECT.infer(spelled).name() == "RegExp"
whole = store.registered("Expressions.TypeScript.TypeScript5.Expression")  # round-tripped in its holder
for encode, decode in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(decode(whole, encode(whole, F.holder_of(spelled))).root, spelled)
# A constant written with its regexp alone, as code writes one: printed from it
written = ts(lambda e: e.constant(lambda c: c.regexp(slash)))
assert TS.DIALECT.validate(written) == [] and TSText.ToText(written) == "/a/g"
assert TSText.ToText(ts(lambda e: e.constant(lambda c: c.regexp(every)))) == EVERY
# Its checks: its regexp's problems too, by its own dialect; a singleton holds none
for spec, problems in [
    (lambda c: c.pattern("a(").regexp(lambda r: r.regexpliteral(lambda l: l)), ["regexp: a regexpliteral needs a pattern"]),
    (lambda c: c.singleton("null").regexp(slash), ["a constant holds a singleton or a regular expression, not both"]),
    (lambda c: c.flags("g"), ["a regular expression needs a pattern"]),
    (lambda c: c.pattern("a").flags("gg"), ["a regular expression's flags cannot be 'gg'"]),
]:
    assert TS.DIALECT.validate(ts(lambda e, spec=spec: e.constant(spec))) == problems
# Evaluation refuses a constant, as TypeScript5's evaluator does any kind but a literal; a string given to `new RegExp`
# is a string, not a term of this dialect
with raises(NotImplementedError, match="TypeScript5 constant is not evaluated"):
    TV.OfAny(spelled)
built = ts(lambda e: e.newexpression(lambda n: n.callee(lambda c: c.identifier(lambda i: i.name("RegExp"))).arguments([
    lambda a: a.literal(lambda li: li.value(lambda v: v.as_scalar("a+")))])))
assert {F.dialect_of(node) for node in F.walk(built)} == {TS.DIALECT.union}

<a id="TSR-07"></a>
## TSR-07 · TypeScript5's `constant` holds a literal's regular expression, its `regexp`, beside its spelling: validated with it, printed from it where it has no spelling, read and written by its form